# 기상청 단기·중기예보 API 호출 예제

이 노트북은 `기상청_단기예보_중기예보_API_발급방법_요약.md`를 기준으로,
공공데이터포털의 **기상청_단기예보 조회서비스**와 **기상청_중기예보 조회서비스**를 Python으로 호출하는 예제입니다.

1. `C:\env\.env`에서 API 키 읽기 (`KMA_SHORT_TERM_KEY`, `KMA_MID_TERM_KEY`)
2. 단기예보: 초단기실황(`getUltraSrtNcst`) + 단기예보(`getVilageFcst`)
3. 중기예보: 중기전망(`getMidFcst`) + 중기육상예보(`getMidLandFcst`) + 중기기온(`getMidTa`)
4. 결과를 pandas DataFrame으로 정리

> 두 API는 **별도 서비스**이므로 인증키도 분리해서 관리합니다.  
> API Key는 절대 노트북에 하드코딩하지 않습니다. 키 값도 출력하지 않습니다.

## 0. 환경 준비

필요 패키지: `requests`, `pandas`, `python-dotenv`

```text
pip install requests pandas python-dotenv
```

In [1]:
import os
from datetime import datetime, timedelta
from urllib.parse import unquote
from zoneinfo import ZoneInfo

import pandas as pd
import requests
from dotenv import load_dotenv

KST = ZoneInfo("Asia/Seoul")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_colwidth", 80)

## 1. `.env`에서 API 키 읽기

`C:\env\.env`에 아래처럼 저장되어 있어야 합니다.

```text
KMA_SHORT_TERM_KEY=단기예보_API_인증키
KMA_MID_TERM_KEY=중기예보_API_인증키
```

두 키는 공공데이터포털에서 **각각 활용신청**한 뒤 발급받습니다.

In [2]:
ENV_PATH = r"C:\env\.env"

load_dotenv(ENV_PATH)

KMA_SHORT_TERM_KEY = os.getenv("KMA_SHORT_TERM_KEY")
KMA_MID_TERM_KEY = os.getenv("KMA_MID_TERM_KEY")

if not KMA_SHORT_TERM_KEY:
    raise ValueError(f"KMA_SHORT_TERM_KEY를 찾을 수 없습니다. {ENV_PATH} 파일을 확인하세요.")
if not KMA_MID_TERM_KEY:
    raise ValueError(f"KMA_MID_TERM_KEY를 찾을 수 없습니다. {ENV_PATH} 파일을 확인하세요.")

print("KMA_SHORT_TERM_KEY 로드 완료 (키 값은 출력하지 않습니다)")
print("KMA_MID_TERM_KEY 로드 완료 (키 값은 출력하지 않습니다)")

KMA_SHORT_TERM_KEY 로드 완료 (키 값은 출력하지 않습니다)
KMA_MID_TERM_KEY 로드 완료 (키 값은 출력하지 않습니다)


## 2. 공통 호출 함수

공공데이터포털 인증키는 URL 인코딩된 상태로 저장되는 경우가 있습니다.
`unquote`로 한 번 디코딩한 뒤 `requests`가 다시 인코딩하도록 하여 **이중 인코딩**을 피합니다.

In [3]:
def call_kma_api(url: str, service_key: str, extra_params: dict) -> dict:
    """기상청 공공데이터 API를 JSON으로 호출하고 response 본문을 반환합니다."""
    params = {
        "serviceKey": unquote(service_key),
        "pageNo": "1",
        "numOfRows": "1000",
        "dataType": "JSON",
        **extra_params,
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    try:
        payload = response.json()
    except ValueError as exc:
        preview = response.text[:300]
        raise ValueError(
            "JSON이 아닌 응답입니다. 인증키 또는 요청 파라미터를 확인하세요.\n"
            f"응답 미리보기: {preview}"
        ) from exc

    header = payload.get("response", {}).get("header", {})
    result_code = header.get("resultCode")
    result_msg = header.get("resultMsg")
    if result_code != "00":
        raise RuntimeError(f"기상청 API 오류: [{result_code}] {result_msg}")

    return payload["response"]


def extract_items(response_body: dict) -> list[dict]:
    items = response_body.get("body", {}).get("items", {}).get("item", [])
    if isinstance(items, dict):
        return [items]
    return items

## 3. 단기예보 조회서비스

| 구분 | 내용 |
|---|---|
| API | 기상청_단기예보 조회서비스 |
| 인증키 | `KMA_SHORT_TERM_KEY` |
| 기본 URL | `https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0` |
| 위치 | 위·경도가 아니라 **격자 좌표 `nx`, `ny`** |
| 용도 | 현재 실황 ~ 글피까지 1시간 단위 상세 예보 |

예제 지점은 **서울 종로(격자 60, 127)** 입니다. 다른 지역은 공공데이터포털 활용가이드의 격자 엑셀을 참고하세요.

In [4]:
SHORT_TERM_BASE = "https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0"

# 서울 종로구(서울시청 인근) 격자. 다른 지역은 활용가이드 격자표 참고
NX, NY = 60, 127
# LOCATION_NAME = "서울 종로"
LOCATION_NAME = "전남 광주"

SHORT_TERM_CATEGORY = {
    "TMP": "기온(℃)",
    "TMN": "최저기온(℃)",
    "TMX": "최고기온(℃)",
    "POP": "강수확률(%)",
    "PTY": "강수형태",
    "PCP": "1시간 강수량",
    "SNO": "1시간 신적설",
    "SKY": "하늘상태",
    "REH": "습도(%)",
    "WSD": "풍속(m/s)",
    "VEC": "풍향(deg)",
    "UUU": "동서바람성분(m/s)",
    "VVV": "남북바람성분(m/s)",
    "WAV": "파고(m)",
}

SKY_CODE = {"1": "맑음", "3": "구름많음", "4": "흐림"}
PTY_CODE = {
    "0": "없음",
    "1": "비",
    "2": "비/눈",
    "3": "눈",
    "4": "소나기",
    "5": "빗방울",
    "6": "빗방울눈날림",
    "7": "눈날림",
}

NCS_T_CATEGORY = {
    "T1H": "기온(℃)",
    "RN1": "1시간 강수량(mm)",
    "UUU": "동서바람성분(m/s)",
    "VVV": "남북바람성분(m/s)",
    "REH": "습도(%)",
    "PTY": "강수형태",
    "VEC": "풍향(deg)",
    "WSD": "풍속(m/s)",
}


def latest_vilage_base(now: datetime | None = None) -> tuple[str, str]:
    """단기예보 최신 발표시각. 02/05/08/11/14/17/20/23시, 발표 10분 후부터 조회."""
    now = now or datetime.now(KST)
    hours = [2, 5, 8, 11, 14, 17, 20, 23]
    candidates = []
    for hour in hours:
        announced = now.replace(hour=hour, minute=10, second=0, microsecond=0)
        candidates.append((announced, f"{hour:02d}00"))

    available = [(t, bt) for t, bt in candidates if now >= t]
    if available:
        base_dt, base_time = available[-1]
        return base_dt.strftime("%Y%m%d"), base_time

    yesterday = now - timedelta(days=1)
    return yesterday.strftime("%Y%m%d"), "2300"


def latest_ncst_base(now: datetime | None = None) -> tuple[str, str]:
    """초단기실황 최신 발표시각. 매시 정시, 약 10분 후부터 조회."""
    now = now or datetime.now(KST)
    base = now.replace(minute=0, second=0, microsecond=0)
    if now.minute < 10:
        base -= timedelta(hours=1)
    return base.strftime("%Y%m%d"), base.strftime("%H00")


vilage_date, vilage_time = latest_vilage_base()
ncst_date, ncst_time = latest_ncst_base()
print(f"조회 지점: {LOCATION_NAME} (nx={NX}, ny={NY})")
print(f"초단기실황 base: {ncst_date} {ncst_time}")
print(f"단기예보   base: {vilage_date} {vilage_time}")

조회 지점: 전남 광주 (nx=60, ny=127)
초단기실황 base: 20260831 1400
단기예보   base: 20260831 1400


### 3-1. 초단기실황 (`getUltraSrtNcst`)

선택한 격자의 **현재에 가까운 실황값**을 조회합니다.

In [5]:
ncst_response = call_kma_api(
    f"{SHORT_TERM_BASE}/getUltraSrtNcst",
    KMA_SHORT_TERM_KEY,
    {
        "base_date": ncst_date,
        "base_time": ncst_time,
        "nx": NX,
        "ny": NY,
    },
)

ncst_df = pd.DataFrame(extract_items(ncst_response))
ncst_df["항목"] = ncst_df["category"].map(NCS_T_CATEGORY).fillna(ncst_df["category"])
ncst_df["값"] = ncst_df.apply(
    lambda row: PTY_CODE.get(str(row["obsrValue"]), row["obsrValue"])
    if row["category"] == "PTY"
    else row["obsrValue"],
    axis=1,
)

print(f"[{LOCATION_NAME}] 초단기실황 {ncst_date} {ncst_time}")
ncst_df[["항목", "category", "값", "baseDate", "baseTime", "nx", "ny"]]

[전남 광주] 초단기실황 20260831 1400


,항목,category,값,baseDate,baseTime,nx,ny
0,강수형태,PTY,비,20260831,1400,60,127
1,습도(%),REH,91,20260831,1400,60,127
2,1시간 강수량(mm),RN1,2.4,20260831,1400,60,127
3,기온(℃),T1H,24.4,20260831,1400,60,127
4,동서바람성분(m/s),UUU,-1.2,20260831,1400,60,127
5,풍향(deg),VEC,73,20260831,1400,60,127
6,남북바람성분(m/s),VVV,-0.3,20260831,1400,60,127
7,풍속(m/s),WSD,1.3,20260831,1400,60,127


### 3-2. 단기예보 (`getVilageFcst`)

발표 시각 기준 **오늘~글피**의 1시간 단위 예보를 조회합니다.
기온, 강수확률, 하늘상태 등 농업 의사결정에 바로 쓸 수 있는 항목을 표로 정리합니다.

In [6]:
vilage_response = call_kma_api(
    f"{SHORT_TERM_BASE}/getVilageFcst",
    KMA_SHORT_TERM_KEY,
    {
        "base_date": vilage_date,
        "base_time": vilage_time,
        "nx": NX,
        "ny": NY,
    },
)

vilage_raw = pd.DataFrame(extract_items(vilage_response))
print(f"[{LOCATION_NAME}] 단기예보 {vilage_date} {vilage_time} 발표, 항목 수: {len(vilage_raw)}")
vilage_raw.head()

[전남 광주] 단기예보 20260831 1400 발표, 항목 수: 798


,baseDate,baseTime,category,fcstDate,fcstTime,fcstValue,nx,ny
0,20260831,1400,TMP,20260831,1500,25,60,127
1,20260831,1400,UUU,20260831,1500,-1.7,60,127
2,20260831,1400,VVV,20260831,1500,0,60,127
3,20260831,1400,VEC,20260831,1500,90,60,127
4,20260831,1400,WSD,20260831,1500,1.8,60,127


In [7]:
MAIN_CATEGORIES = ["TMP", "TMN", "TMX", "POP", "PTY", "PCP", "SKY", "REH", "WSD"]

hourly = (
    vilage_raw[vilage_raw["category"].isin(MAIN_CATEGORIES)]
    .pivot_table(
        index=["fcstDate", "fcstTime"],
        columns="category",
        values="fcstValue",
        aggfunc="first",
    )
    .reset_index()
)

hourly = hourly.rename(columns=SHORT_TERM_CATEGORY)
hourly = hourly.rename(columns={"fcstDate": "예보일자", "fcstTime": "예보시각"})
hourly = hourly.sort_values(["예보일자", "예보시각"]).reset_index(drop=True)

if "하늘상태" in hourly.columns:
    hourly["하늘상태"] = hourly["하늘상태"].map(lambda v: SKY_CODE.get(str(v), v))
if "강수형태" in hourly.columns:
    hourly["강수형태"] = hourly["강수형태"].map(lambda v: PTY_CODE.get(str(v), v))

print(f"시간별 예보 {len(hourly)}건 (오늘~글피)")
hourly

시간별 예보 66건 (오늘~글피)


category,예보일자,예보시각,1시간 강수량,강수확률(%),강수형태,습도(%),하늘상태,최저기온(℃),기온(℃),최고기온(℃),풍속(m/s)
0,20260831,1500,3.0mm,60,비,90,흐림,NaN,25,NaN,1.8
1,20260831,1600,2.0mm,60,비,90,흐림,NaN,25,NaN,1.6
2,20260831,1700,3.0mm,60,비,90,흐림,NaN,24,NaN,1.4
3,20260831,1800,3.0mm,60,비,90,흐림,NaN,24,NaN,1.6
4,20260831,1900,3.0mm,60,비,90,흐림,NaN,24,NaN,1.6
...,...,...,...,...,...,...,...,...,...,...,...
61,20260903,1200,0,10,없음,60,구름많음,NaN,28,NaN,1
62,20260903,1500,0,10,없음,55,구름많음,NaN,30,30.0,1
63,20260903,1800,0,10,없음,60,구름많음,NaN,28,NaN,1
64,20260903,2100,0,20,없음,70,구름많음,NaN,25,NaN,1


## 4. 중기예보 조회서비스

| 구분 | 내용 |
|---|---|
| API | 기상청_중기예보 조회서비스 |
| 인증키 | `KMA_MID_TERM_KEY` |
| 기본 URL | `https://apis.data.go.kr/1360000/MidFcstInfoService` |
| 위치 | 격자 좌표가 아니라 **예보구역코드 `regId` / 지점번호 `stnId`** |
| 발표 | 하루 2회 (06시, 18시), 최근 24시간 자료만 제공 |
| 용도 | 단기예보 이후 **최대 11일** 전망 |

주의: **육상예보용 `regId`와 기온예보용 `regId`가 다릅니다.**

- 중기육상: 서울·인천·경기 `11B00000` (광역)
- 중기기온: 서울 `11B10101` (도시)

In [8]:
MID_TERM_BASE = "https://apis.data.go.kr/1360000/MidFcstInfoService"

# 중기전망(getMidFcst) 지점번호
STN_ID = "109"  # 서울·인천·경기도

# 중기육상예보 구역코드 (광역)
LAND_REG_ID = "11B00000"  # 서울, 인천, 경기도

# 중기기온예보 구역코드 (도시)
TA_REG_ID = "11B10101"  # 서울

LAND_REGIONS = {
    "11B00000": "서울·인천·경기도",
    "11D10000": "강원도영서",
    "11D20000": "강원도영동",
    "11C20000": "대전·세종·충청남도",
    "11C10000": "충청북도",
    "11F20000": "광주·전라남도",
    "11F10000": "전북",
    "11H10000": "대구·경상북도",
    "11H20000": "부산·울산·경상남도",
    "11G00000": "제주도",
}

TA_REGIONS = {
    "11B10101": "서울",
    "11B20201": "인천",
    "11B20601": "수원",
    "11B20305": "파주",
    "11D10301": "춘천",
    "11D20501": "강릉",
    "11C20401": "대전",
    "11C10301": "청주",
    "11F20501": "광주",
    "11F10201": "전주",
    "11H20201": "부산",
    "11H10701": "대구",
    "11G00201": "제주",
}


def latest_mid_tmfc(now: datetime | None = None) -> str:
    """중기예보 최신 발표시각. 06시/18시, 발표 10분 후부터 조회. YYYYMMDDHHMM"""
    now = now or datetime.now(KST)
    today_06 = now.replace(hour=6, minute=10, second=0, microsecond=0)
    today_18 = now.replace(hour=18, minute=10, second=0, microsecond=0)

    if now >= today_18:
        return now.strftime("%Y%m%d") + "1800"
    if now >= today_06:
        return now.strftime("%Y%m%d") + "0600"

    yesterday = now - timedelta(days=1)
    return yesterday.strftime("%Y%m%d") + "1800"


tm_fc = latest_mid_tmfc()
print(f"중기예보 발표시각 tmFc: {tm_fc}")
print(f"육상구역: {LAND_REGIONS[LAND_REG_ID]} ({LAND_REG_ID})")
print(f"기온구역: {TA_REGIONS[TA_REG_ID]} ({TA_REG_ID})")

중기예보 발표시각 tmFc: 202608310600
육상구역: 서울·인천·경기도 (11B00000)
기온구역: 서울 (11B10101)


### 4-1. 중기전망 (`getMidFcst`)

지점번호(`stnId`)로 **텍스트 기상전망**을 조회합니다.

In [9]:
mid_fcst_response = call_kma_api(
    f"{MID_TERM_BASE}/getMidFcst",
    KMA_MID_TERM_KEY,
    {"stnId": STN_ID, "tmFc": tm_fc},
)

mid_fcst_items = extract_items(mid_fcst_response)
outlook_text = mid_fcst_items[0].get("wfSv", "") if mid_fcst_items else ""

print(f"[중기전망] stnId={STN_ID}, tmFc={tm_fc}")
print("-" * 60)
print(outlook_text)

[중기전망] stnId=109, tmFc=202608310600
------------------------------------------------------------
○ (강수) 이번 예보기간은 구름많겠습니다.
○ (기온) 아침 기온은 18~22℃, 낮 기온은 28~30℃로 평년(최저기온 16~20℃, 최고기온 26~29℃)과 비슷하거나 조금 높겠습니다.
○ (해상) 서해중부해상의 물결은 0.5~2.0m로 일겠습니다.
○ (주말전망) 9월 5일(토)~6일(일)은 구름많겠습니다. 아침 기온은 18~22℃, 낮 기온은 28~29℃가 되겠습니다.

* 이번 예보기간에는 최고체감온도가 31℃ 안팎으로 오르는 곳이 있겠으니 건강관리에 각별히 유의하기 바랍니다.


### 4-2. 중기육상예보 (`getMidLandFcst`)

광역 구역의 **날씨(wf)와 강수확률(rnSt)** 을 조회합니다.
3~7일은 오전/오후, 8~10일은 하루 단위입니다.

In [10]:
land_response = call_kma_api(
    f"{MID_TERM_BASE}/getMidLandFcst",
    KMA_MID_TERM_KEY,
    {"regId": LAND_REG_ID, "tmFc": tm_fc},
)

land_item = extract_items(land_response)[0]
print(f"[중기육상예보] {LAND_REGIONS[LAND_REG_ID]} ({LAND_REG_ID})")
pd.DataFrame([land_item]).T.rename(columns={0: "값"})

[중기육상예보] 서울·인천·경기도 (11B00000)


,값
regId,11B00000
rnSt4Am,20
rnSt4Pm,20
rnSt5Am,20
rnSt5Pm,20
rnSt6Am,20
rnSt6Pm,20
rnSt7Am,20
rnSt7Pm,20
rnSt8,20


### 4-3. 중기기온 (`getMidTa`)

도시 단위 **최저기온(taMin) / 최고기온(taMax)** 을 조회합니다.

In [11]:
ta_response = call_kma_api(
    f"{MID_TERM_BASE}/getMidTa",
    KMA_MID_TERM_KEY,
    {"regId": TA_REG_ID, "tmFc": tm_fc},
)

ta_item = extract_items(ta_response)[0]
print(f"[중기기온] {TA_REGIONS[TA_REG_ID]} ({TA_REG_ID})")
pd.DataFrame([ta_item]).T.rename(columns={0: "값"})

[중기기온] 서울 (11B10101)


,값
regId,11B10101
taMin4,21
taMin4Low,1
taMin4High,1
taMax4,29
taMax4Low,1
taMax4High,1
taMin5,20
taMin5Low,1
taMin5High,1


### 4-4. 중기 육상 + 기온 합치기

발표일 기준 **+3일 ~ +10일** 전망을 하루 단위 표로 정리합니다.

In [12]:
announce_dt = datetime.strptime(tm_fc, "%Y%m%d%H%M").replace(tzinfo=KST)
rows = []

for day in range(3, 11):
    forecast_date = (announce_dt + timedelta(days=day)).strftime("%Y-%m-%d")
    row = {"일차": f"+{day}일", "날짜": forecast_date}

    if day <= 7:
        row["날씨(오전)"] = land_item.get(f"wf{day}Am")
        row["날씨(오후)"] = land_item.get(f"wf{day}Pm")
        row["강수확률(오전)"] = land_item.get(f"rnSt{day}Am")
        row["강수확률(오후)"] = land_item.get(f"rnSt{day}Pm")
    else:
        row["날씨(오전)"] = land_item.get(f"wf{day}")
        row["날씨(오후)"] = land_item.get(f"wf{day}")
        row["강수확률(오전)"] = land_item.get(f"rnSt{day}")
        row["강수확률(오후)"] = land_item.get(f"rnSt{day}")

    row["최저기온(℃)"] = ta_item.get(f"taMin{day}")
    row["최고기온(℃)"] = ta_item.get(f"taMax{day}")
    rows.append(row)

mid_df = pd.DataFrame(rows)
print(f"중기예보 요약 | 육상: {LAND_REGIONS[LAND_REG_ID]} / 기온: {TA_REGIONS[TA_REG_ID]}")
mid_df

중기예보 요약 | 육상: 서울·인천·경기도 / 기온: 서울


,일차,날짜,날씨(오전),날씨(오후),강수확률(오전),강수확률(오후),최저기온(℃),최고기온(℃)
0,+3일,2026-09-03,None,None,NaN,NaN,NaN,NaN
1,+4일,2026-09-04,구름많음,구름많음,20.0,20.0,21.0,29.0
2,+5일,2026-09-05,구름많음,구름많음,20.0,20.0,20.0,28.0
3,+6일,2026-09-06,구름많음,구름많음,20.0,20.0,20.0,28.0
4,+7일,2026-09-07,구름많음,구름많음,20.0,20.0,20.0,29.0
5,+8일,2026-09-08,구름많음,구름많음,20.0,20.0,20.0,29.0
6,+9일,2026-09-09,구름많음,구름많음,20.0,20.0,20.0,29.0
7,+10일,2026-09-10,구름많음,구름많음,20.0,20.0,20.0,29.0


## 5. 농업 Agent 활용 포인트

문서에서 안내한 것처럼, AI Agent에서는 **단기 Tool과 중기 Tool을 분리**하는 것이 핵심입니다.

```text
사용자: "앞으로 10일간 날씨를 분석해서 토마토 농작업 계획을 세워줘."
         │
         ▼
    AI Agent
    ┌────────┴────────┐
    ▼                 ▼
단기예보 Tool       중기예보 Tool
KMA_SHORT_TERM_KEY  KMA_MID_TERM_KEY
1~3일 상세          최대 11일 전망
```

| Tool | 함수 예시 | 키 | 위치 파라미터 |
|---|---|---|---|
| 단기 | `get_short_term_weather(nx, ny)` | `KMA_SHORT_TERM_KEY` | 격자 `nx`, `ny` |
| 중기 | `get_mid_term_weather(reg_id)` | `KMA_MID_TERM_KEY` | 육상/기온 `regId` |

다음 실습에서는 위 호출을 LangChain/LangGraph `@tool`로 감싸 Agent가 날짜 범위에 따라 알아서 선택하도록 구성하면 됩니다.